# Data Transformation

## Install Dependencies

In [1]:
!chmod +x install_packages.sh farsite/lcpmake
!./install_packages.sh

## Imports and Environment Setup

In [10]:
import sys
import io
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import requests
import json
import geojson
from datetime import datetime
import time
import shapely
from shapely.geometry import box, Point, Polygon
import matplotlib.pyplot as plt
from pyproj import Transformer
import zipfile
import subprocess
from io import StringIO
import contextily as ctx
from osgeo import gdal, ogr, osr

# Enable GDAL exceptions
gdal.UseExceptions()
osr.UseExceptions()

from pathlib import Path

BASE_DIR  = Path().parent
FARSITE_DIR = BASE_DIR / 'farsite'

from farsite import *

## Define Ignition Constraints

In [11]:
IGNITION_LAT = 38.9014
IGNITION_LON = -120.0306
FIREMAP_USERNAME = 'YOUR_USERNAME'  ## Change to your Firemap account username
FIREMAP_PASSWORD = 'YOUR_PASSWORD'  ## Change to your Firemap account password

## Compile Landscape Data
Given landscape data in `TIF` and `GeoJSON` format, use FARSITE's ``lcpmake`` executable to compile and ``LCP`` landscape file formatted for FARSITE compatibility. 

Eight landscape raster bands are required to run FARSITE: slope, aspect, fire behavior fuel model, tree canopy cover, canopy height, canopy base height, and canopy bulk density.

### Confirm paths to raster bands

In [12]:
TIFF_DIR = BASE_DIR / 'Forest'
LCPMAKE_EXEC_PATH = FARSITE_DIR / 'lcpmake'

# Check that input data exists
print("Input data locations:")
if not os.path.exists(TIFF_DIR):
    print("WARNING: Missing landscape TIFF files.")
else:
    print("Landscape layers (.tif) directory: ", TIFF_DIR)
    
if not os.path.exists(LCPMAKE_EXEC_PATH):
    print("WARNING: Missing LCPMAKE executable to compile .LCP file.")
else:
    print("lcpmake executable path: ", LCPMAKE_EXEC_PATH)

Input data locations:
Landscape layers (.tif) directory:  Forest
lcpmake executable path:  farsite/lcpmake


### Create output location for transformed landscape data

In [13]:
LCP_OUTPUT_DIR = BASE_DIR / 'Forest_LCP_Outputs'
LCP_OUTPUT_PATH = LCP_OUTPUT_DIR / 'landscape.lcp'

# Create output DIRECTORY if not already exist
Path(LCP_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)  # Create output directory if diesn't exist

print("\nOutput data locations:")
print("ASCII files (.asc) and landscape file (.lcp) output directory: ", LCP_OUTPUT_PATH)
print("Landscape filepath ", LCP_OUTPUT_PATH)


Output data locations:
ASCII files (.asc) and landscape file (.lcp) output directory:  Forest_LCP_Outputs/landscape.lcp
Landscape filepath  Forest_LCP_Outputs/landscape.lcp


### Identify landscape data filepaths

In [14]:
# Raster band paths
ELEV_PATH = TIFF_DIR / 'Forest_elevation.tif'
FBFM_PATH = TIFF_DIR / 'Forest_fbfm13.tif'
TREELIST_PATH = TIFF_DIR / 'Forest_Treelist.json'

print("Given landscape layers:")
print("Elevation: ", ELEV_PATH)
print("Fire Behavior Fuel Model: ", FBFM_PATH)
print('Tree data (Canopy Height, Canopy Base Height, Canopy Bulk Density): ', TREELIST_PATH)

print("\nMissing layers: Slope, Aspect, Canopy Cover")

Given landscape layers:
Elevation:  Forest/Forest_elevation.tif
Fire Behavior Fuel Model:  Forest/Forest_fbfm13.tif
Tree data (Canopy Height, Canopy Base Height, Canopy Bulk Density):  Forest/Forest_Treelist.json

Missing layers: Slope, Aspect, Canopy Cover


### Define function to convert existing `.tif` data to `.asc`

In [15]:
def convert_tif_to_ascii(
    input_path: str | Path,
    output_path: str | Path | None = None,
    band: int = 1,
    verbose: bool = False,
) -> Path:
    """
    Convert a single-band raster TIFF to an ASCII grid (.asc) file using GDAL.

    Args:
        input_path:  Path to the source .tif file.
        output_path: Path for the output .asc file. If None, uses the same
                     directory and stem as the input (e.g. slope.tif -> slope.asc).
        band:        Band index to export (1-based, default 1).
        verbose:     Print progress messages.

    Returns:
        Path to the written .asc file.

    Raises:
        FileNotFoundError: If the input file does not exist.
        RuntimeError:      If GDAL fails to open or translate the file.
    """
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    if output_path is None:
        output_path = input_path.with_suffix(".asc")
    output_path = Path(output_path)

    ds = gdal.Open(str(input_path))
    if ds is None:
        raise RuntimeError(f"GDAL could not open: {input_path}")

    n_bands = ds.RasterCount
    if band < 1 or band > n_bands:
        raise ValueError(f"Band {band} out of range — file has {n_bands} band(s).")
    ds = None  # close before Translate

    result = gdal.Translate(
        str(output_path),
        str(input_path),
        format="AAIGrid",
        bandList=[band],
    )

    if result is None:
        raise RuntimeError(f"GDAL Translate failed for: {input_path}")
    result = None  # flush and close

    if verbose:
        print(f"  ✓ {output_path.name}")

    return output_path

In [16]:
# Call function to convert TIFF rasters to ASCII
ELEV_ASCII = convert_tif_to_ascii(input_path=str(ELEV_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_elevation.asc"), verbose=True)
FBFM_ASCII = convert_tif_to_ascii(input_path=str(FBFM_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_fbfm13.asc"), verbose=True)

  ✓ Forest_elevation.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


  ✓ Forest_fbfm13.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


### Use GDAL to derive Slope and Aspect layers from Elevation raster band

In [29]:
# Get Slope aand Aspect from Elevation TIF
SLPP_PATH = LCP_OUTPUT_DIR / "slope.tif"
ASP_PATH = LCP_OUTPUT_DIR / "aspect.tif"

gdal.DEMProcessing(SLPP_PATH, str(ELEV_PATH), "slope", slopeFormat="percent")
gdal.DEMProcessing(LCP_OUTPUT_DIR / "aspect.tif", str(ELEV_PATH), "aspect")

Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_name: Open of /opt/conda/share/proj failed
Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_name: Open of /opt/conda/share/proj failed


<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x7799e9f96340> >

### Use GDAL to rasterize Canopy Height, Base Height, Bulk Density, and Diameter from Tree Data GeoJSON

In [32]:
# Output paths
HT_PATH = LCP_OUTPUT_DIR / "canopy_height.tif"
CBH_PATH = LCP_OUTPUT_DIR / "canopy_base_height.tif"
CBD_PATH = LCP_OUTPUT_DIR / "canopy_bulk_density.tif"
DIA_PATH = LCP_OUTPUT_DIR / "maximum_canopy_diameter.tif"

In [33]:
tree_layers = {
    HT_PATH: "HT",
    CBH_PATH:  "CBH",
    CBD_PATH: "CBD",
    DIA_PATH: "DIA"
}


# GeoTIFF bounds, height, and width should be the same as given .tif files
ref_ds = gdal.Open(FBFM_PATH)
gt = ref_ds.GetGeoTransform()
nx = ref_ds.RasterXSize
ny = ref_ds.RasterYSize
ref_ds = None

x_min = gt[0]
y_max = gt[3]
x_max = x_min + nx * gt[1]
y_min = y_max + ny * gt[5]  # gt[5] is negative


# Rasterize each tree data layer
for filepath, field in tree_layers.items():
    gdal.Rasterize(
        str(filepath),
        str(TREELIST_PATH),
        format="GTiff",
        attribute=field,
        outputBounds=[x_min, y_min, x_max, y_max],
        width=nx,
        height=ny,
        noData=9999,
        outputType=gdal.GDT_Float32,
    )

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed
ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


### Calculate Canopy Cover (CC)
Compute Canopy Cover (CC) from canopy diameter data:
1. Rasterize Maximum Canopy Diameter data from `Forest_Treelist.json` 
2. Convert Canopy Diameter to Canopy Cover

In [28]:
CC_PATH = LCP_OUTPUT_DIR / "canopy_cover.tif"


# Rasterize Canopy Diameter
gdal.Rasterize(
    str(CC_PATH),
    str(TREELIST_PATH),
    format="GTiff",
    attribute="DIA",
    outputBounds=[x_min, y_min, x_max, y_max],
    width=nx,
    height=ny,
    noData=9999,
    outputType=gdal.GDT_Float32,
)


# Convert diameter to canopy cover percentage
px_w = gt[1]
px_h = -gt[5]
cell_area = px_w * px_h

ds = gdal.Open(str(CC_PATH), gdal.GA_Update)
band = ds.GetRasterBand(1)
data = band.ReadAsArray().astype(np.float32)

valid = data != -9999
data[valid] = np.clip(
    (np.pi * (data[valid] / 2) ** 2) / cell_area * 100.0,
    0.0, 100.0
)

band.WriteArray(data)
band.FlushCache()
ds = None

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


### Convert remaining `.tif` files to `asc`

In [40]:
SLPP_ASCII = convert_tif_to_ascii(input_path=str(SLPP_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_slope.asc"), verbose=True)
ASP_ASCII = convert_tif_to_ascii(input_path=str(ASP_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_aspect.asc"), verbose=True)
HT_ASCII = convert_tif_to_ascii(input_path=str(HT_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_canopyHeight.asc"), verbose=True) 
CBH_ASCII = convert_tif_to_ascii(input_path=str(CBH_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_canopyBaseHeight.asc"), verbose=True)
CBD_ASCII = convert_tif_to_ascii(input_path=str(CBD_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_canopyBulkDensity.asc"), verbose=True)
CC_ASCII = convert_tif_to_ascii(input_path=str(CC_PATH), output_path=str(LCP_OUTPUT_DIR / "Forest_canopyCover.asc"), verbose=True)

  ✓ Forest_slope.asc
  ✓ Forest_aspect.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


  ✓ Forest_canopyHeight.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


  ✓ Forest_canopyBaseHeight.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


  ✓ Forest_canopyBulkDensity.asc
  ✓ Forest_canopyCover.asc


Warning 1: PROJ: proj_create_from_database: Open of /opt/conda/share/proj failed


In [44]:
print("Landscape raster bands in .tif format")
print("-------------------------------------")
print("Elevation: ", ELEV_ASCII)
print("Slope: ", SLPP_ASCII)
print("Aspect: ", ASP_ASCII)
print("Fire Behavior Fuel Model: ", FBFM_ASCII)
print("Canopy Height: ", HT_ASCII)
print("Canopy Base Height: ", CBH_ASCII)
print("Canopy Bulk Density: ", CBD_ASCII)
print("Canopy Cover: ", CC_ASCII)

Landscape raster bands in .tif format
-------------------------------------
Elevation:  Forest_LCP_Outputs/Forest_elevation.asc
Fire Behavior Fuel Model:  Forest_LCP_Outputs/Forest_fbfm13.asc
Slope:  Forest_LCP_Outputs/Forest_slope.asc
Aspect:  Forest_LCP_Outputs/Forest_aspect.asc
Canopy Height:  Forest_LCP_Outputs/Forest_canopyHeight.asc
Canopy Base Height:  Forest_LCP_Outputs/Forest_canopyBaseHeight.asc
Canopy Bulk Density:  Forest_LCP_Outputs/Forest_canopyBulkDensity.asc
Canopy Cover:  Forest_LCP_Outputs/Forest_canopyCover.asc


## Compile `LCP` file using `lcpmake` executable

In [52]:
lcp_filename = LCP_OUTPUT_DIR / 'landscape'

# Command to generate landscape file
cmd = [
        str(LCPMAKE_EXEC_PATH),
        "-latitude", str(IGNITION_LAT),
        "-landscape", str(lcp_filename),
        "-elevation", str(ELEV_ASCII),
        "-slope", str(SLPP_ASCII),
        "-aspect", str(ASP_ASCII),
        "-fuel", str(FBFM_ASCII),
        "-cover", str(CC_ASCII),
        "-height", str(HT_ASCII),
        "-base", str(CBH_ASCII),
        "-density", str(CBD_ASCII),
    ]
cmd.append('-fbfm13')


# Print command
print("\nRunning lcpmake command:")
print(" ".join(cmd))


# Run command to generate .lcp file
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    raise RuntimeError(
        f"lcpmake failed with return code {result.returncode}\n"
        f"stdout: {result.stdout}\nstderr: {result.stderr}"
    )
    
lcp_filepath = lcp_filename.with_suffix('.lcp')

print(f"\n✓ Landscape file created: {lcp_filepath}")
print(f"  Size: {lcp_filepath.stat().st_size / 1024:.1f} KB")


Running lcpmake command:
farsite/lcpmake -latitude 38.9014 -landscape Forest_LCP_Outputs/landscape -elevation Forest_LCP_Outputs/Forest_elevation.asc -slope Forest_LCP_Outputs/Forest_slope.asc -aspect Forest_LCP_Outputs/Forest_aspect.asc -fuel Forest_LCP_Outputs/Forest_fbfm13.asc -cover Forest_LCP_Outputs/Forest_canopyCover.asc -height Forest_LCP_Outputs/Forest_canopyHeight.asc -base Forest_LCP_Outputs/Forest_canopyBaseHeight.asc -density Forest_LCP_Outputs/Forest_canopyBulkDensity.asc -fbfm13

✓ Landscape file created: Forest_LCP_Outputs/landscape.lcp
  Size: 654.0 KB
